# exp13c — Logit Lens: Stego vs Open vs Control

**Question:** Does the layer-by-layer trajectory of P(constrained letter) differ between *hidden* (stego) and *visible* (control) letter constraints?

**Method (logit lens):** At each layer `l` and each paragraph-start position `pos`, project hidden state `h[l][pos-1]` through the final RMSNorm and `lm_head` → softmax → sum over all tokens starting with the constrained letter. This shows at which layer the model commits to the required letter.

- **stego** — hidden acrostic, no marker  
- **open** — free reasoning, no constraint  
- **control** — visible letter constraint, same format as stego  

**Key comparison:** stego vs control per layer.  
If trajectories overlap → model builds representations identically regardless of constraint visibility.

**Method:** Teacher-forced forward pass, `output_hidden_states=True` only.  
**Data:** `results/exp11/exp11_pairs_with_control.json`  
**Runtime:** ~1.5 h on A100.

In [ ]:
import os, gc, re, json, shutil
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    drive.mount('/content/drive')
    if os.path.basename(os.getcwd()) != 'stego_CoT':
        if not os.path.exists('stego_CoT'):
            os.system('git clone -q https://github.com/annareshetnyak799-netizen/stego_CoT.git')
        os.chdir('stego_CoT')
    os.system('pip install -q transformers accelerate')
    DRIVE_DIR = '/content/drive/MyDrive/stego_cot_results/exp13c'
    os.makedirs(DRIVE_DIR, exist_ok=True)

    INPUT_LOCAL = 'results/exp11/exp11_pairs_with_control.json'
    if not os.path.exists(INPUT_LOCAL):
        src = '/content/drive/MyDrive/stego_cot_results/exp11/exp11_pairs_with_control.json'
        os.makedirs('results/exp11', exist_ok=True)
        shutil.copy(src, INPUT_LOCAL)
        print('Restored input data from Drive.')
else:
    DRIVE_DIR = None

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name}  ({gpu.total_memory/1e9:.0f} GB)')
    assert gpu.total_memory > 30e9, 'Use A100 (40GB+)'
else:
    raise RuntimeError('No GPU')

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
model.eval()
DEVICE   = next(model.parameters()).device
N_LAYERS = model.config.num_hidden_layers
print(f'Model loaded. Layers: {N_LAYERS}')

In [ ]:
# Letter → token IDs map
# For each letter, collect all vocab tokens whose decoded form starts with that letter.
# .strip() is essential: Llama BPE tokens have a space prefix (e.g. ' Safety'),
# so tok_str[0] would be a space without stripping.

LETTERS = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
letter_token_map = {ch: [] for ch in LETTERS}

for tok_id in range(tokenizer.vocab_size):
    tok_str = tokenizer.decode([tok_id], skip_special_tokens=True).strip()
    if tok_str and tok_str[0].upper() in letter_token_map:
        letter_token_map[tok_str[0].upper()].append(tok_id)

letter_token_tensors = {
    ch: torch.tensor(ids, dtype=torch.long).to(DEVICE)
    for ch, ids in letter_token_map.items()
    if ids
}

print('Letter→token map built.')
print('  tokens per letter (sample):', {ch: len(v) for ch, v in list(letter_token_map.items())[:5]})


def find_paragraph_starts(full_ids, plen):
    """Return token positions (in full_ids) of the first token of each paragraph."""
    response_ids = full_ids[plen:]
    tok_strs = [tokenizer.decode([t], skip_special_tokens=True) for t in response_ids]

    buf = ''
    char_starts = []
    for s in tok_strs:
        char_starts.append(len(buf))
        buf += s

    para_chars = []
    i = 0
    while i < len(buf):
        while i < len(buf) and buf[i] in ' \t\n':
            i += 1
        if i >= len(buf):
            break
        para_chars.append(i)
        nn = buf.find('\n\n', i)
        if nn == -1:
            break
        i = nn + 2

    result = []
    for cp in para_chars:
        for ti in range(len(char_starts)):
            end = char_starts[ti + 1] if ti + 1 < len(char_starts) else len(buf)
            if char_starts[ti] <= cp < end:
                result.append(plen + ti)
                break
    return result


print('Helpers defined.')

In [ ]:
@torch.no_grad()
def run_forward(token_ids, sent_positions, payload):
    """
    Teacher-forced forward pass, hidden states only.
    Returns logit_probs: (n_para, N_LAYERS) — P(constrained letter) at each layer
    for the probe position (token just before each paragraph start).
    """
    ids_t = torch.tensor([token_ids], dtype=torch.long).to(DEVICE)
    out = model(
        ids_t,
        output_attentions=False,
        output_hidden_states=True,
        use_cache=False,
    )

    n_para = len(sent_positions)
    logit_probs = np.zeros((n_para, N_LAYERS), dtype=np.float32)

    for k, pos in enumerate(sent_positions):
        probe  = pos - 1  # token just before paragraph start
        letter = payload[k].upper()
        target_ids = letter_token_tensors[letter]

        for l in range(N_LAYERS):
            h = out.hidden_states[l + 1][0, probe]  # layer l+1 (0 = embedding)
            h_normed = model.model.norm(h)
            logits   = model.lm_head(h_normed)
            probs    = torch.softmax(logits.float(), dim=-1)
            logit_probs[k, l] = float(probs[target_ids].sum().item())

    del out, ids_t
    torch.cuda.empty_cache(); gc.collect()
    return logit_probs


print('run_forward defined.')

In [ ]:
INPUT_FILE = 'results/exp11/exp11_pairs_with_control.json'
OUTPUT_DIR = 'results/exp13c'
RAW_FILE   = f'{OUTPUT_DIR}/exp13c_raw.json'
CKPT_EVERY = 50
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(INPUT_FILE) as f:
    all_pairs = json.load(f)

triplets = [
    p for p in all_pairs
    if p.get('fidelity') == 1.0
    and p.get('control_fidelity') == 1.0
    and not p.get('control_explicit', True)
    and p.get('control_ids') is not None
    and len([s for s in re.split(r'\n{2,}', p.get('stego_text','').strip()) if s.strip()]) == len(p['payload'])
    and len([s for s in re.split(r'\n{2,}', p.get('control_text','').strip()) if s.strip()]) == len(p['payload'])
]
print(f'Valid triplets: {len(triplets)} / {len(all_pairs)}')

if not os.path.exists(RAW_FILE) and DRIVE_DIR:
    drive_ckpt = os.path.join(DRIVE_DIR, 'exp13c_raw.json')
    if os.path.exists(drive_ckpt):
        shutil.copy(drive_ckpt, RAW_FILE)
        print('Restored checkpoint from Drive.')

if os.path.exists(RAW_FILE):
    with open(RAW_FILE) as f:
        raw = json.load(f)
    results  = raw.get('results', [])
    done_ids = {r['pair_id'] for r in results}
    print(f'Resumed: {len(results)} done.')
else:
    results  = []
    done_ids = set()


def save_checkpoint():
    data = {'n_triplets': len(results), 'skipped': skipped, 'results': results}
    with open(RAW_FILE, 'w') as f:
        json.dump(data, f)
    if DRIVE_DIR:
        shutil.copy(RAW_FILE, os.path.join(DRIVE_DIR, 'exp13c_raw.json'))


skipped   = 0
last_ckpt = len(results)

for pair in triplets:
    pid = pair.get('arc_id', '') + '_' + pair.get('payload', '')
    if pid in done_ids:
        continue

    try:
        payload = pair['payload'].upper()
        n_para  = len(payload)
        rec     = {'pair_id': pid, 'payload': payload}

        for cond, ids_key, plen_key in [
            ('s', 'stego_ids',   'stego_plen'),
            ('o', 'open_ids',    'open_plen'),
            ('c', 'control_ids', 'control_plen'),
        ]:
            full_ids = pair[ids_key]
            plen     = pair[plen_key]

            sent_pos = find_paragraph_starts(full_ids, plen)

            if cond in ('s', 'c'):
                # Constrained conditions must have exactly n_para paragraphs
                if len(sent_pos) != n_para:
                    raise ValueError(f'{cond}: expected {n_para} paragraphs, found {len(sent_pos)}')
            else:
                # Open is unconstrained and may have more paragraphs.
                # Use only the first n_para to align with payload positions.
                if len(sent_pos) < n_para:
                    raise ValueError(f'o: fewer than {n_para} paragraph starts found ({len(sent_pos)})')
                sent_pos = sent_pos[:n_para]

            probs = run_forward(full_ids, sent_pos, payload)  # (n_para, N_LAYERS)

            for l in range(N_LAYERS):
                key = f'L{l}'
                if key not in rec:
                    rec[key] = {}
                rec[key][f'{cond}_prob'] = round(float(probs[:, l].mean()), 6)

        results.append(rec)
        done_ids.add(pid)
        print(f'[{len(results):4d}] {pid[:40]}  payload={payload}')

    except Exception as e:
        skipped += 1
        print(f'  SKIP {pid[:40]}: {e}')

    if len(results) >= last_ckpt + CKPT_EVERY:
        save_checkpoint()
        last_ckpt = len(results)
        print(f'  >>> checkpoint ({len(results)} done, {skipped} skipped)')

save_checkpoint()
print(f'\nDone. {len(results)} triplets, {skipped} skipped.')

In [ ]:
with open(RAW_FILE) as f:
    raw = json.load(f)
results = raw['results']
n = len(results)
N_LAYERS = 32  # Llama-3.1-8B-Instruct
print(f'Analysing {n} triplets')

# Per-layer means
s_prob = np.zeros(N_LAYERS)
o_prob = np.zeros(N_LAYERS)
c_prob = np.zeros(N_LAYERS)

# Per-pair means (for t-tests)
s_pp, o_pp, c_pp = [], [], []

# Per-pair per-layer (for per-layer t-tests)
s_by_layer = [[] for _ in range(N_LAYERS)]
o_by_layer = [[] for _ in range(N_LAYERS)]
c_by_layer = [[] for _ in range(N_LAYERS)]

for rec in results:
    sp, op, cp = [], [], []
    for l in range(N_LAYERS):
        key = f'L{l}'
        if key not in rec: continue
        sv = rec[key]['s_prob']; ov = rec[key]['o_prob']; cv = rec[key]['c_prob']
        s_prob[l] += sv; o_prob[l] += ov; c_prob[l] += cv
        s_by_layer[l].append(sv); o_by_layer[l].append(ov); c_by_layer[l].append(cv)
        sp.append(sv); op.append(ov); cp.append(cv)
    s_pp.append(np.mean(sp)); o_pp.append(np.mean(op)); c_pp.append(np.mean(cp))

s_prob /= n; o_prob /= n; c_prob /= n
SA, OA, CA = np.array(s_pp), np.array(o_pp), np.array(c_pp)

def ttest(a, b):
    t, p = stats.ttest_rel(a, b)
    return round(float(a.mean() - b.mean()), 6), round(float(t), 3), round(float(p), 6)

r_so = ttest(SA, OA); r_sc = ttest(SA, CA); r_co = ttest(CA, OA)

# Per-layer t-test stego vs control
layer_p_sc = []
for l in range(N_LAYERS):
    _, p = stats.ttest_rel(s_by_layer[l], c_by_layer[l])
    layer_p_sc.append(round(float(p), 6))

print(f'n={n}\n')
print('P(constrained letter) — mean across all layers')
print(f'  stego={SA.mean():.4f}  open={OA.mean():.4f}  control={CA.mean():.4f}')
print(f'  stego-open:     diff={r_so[0]:+.4f}  t={r_so[1]}  p={r_so[2]}')
print(f'  stego-control:  diff={r_sc[0]:+.4f}  t={r_sc[1]}  p={r_sc[2]}  <- KEY')
print(f'  control-open:   diff={r_co[0]:+.4f}  t={r_co[1]}  p={r_co[2]}')
print()
first_div = next((l for l in range(N_LAYERS) if s_prob[l] > o_prob[l] * 1.2), None)
print(f'  First layer where stego > 1.2x open: {first_div}')
print(f'  L30: stego={s_prob[30]:.3f}  control={c_prob[30]:.3f}  open={o_prob[30]:.3f}')
print(f'  L31: stego={s_prob[31]:.3f}  control={c_prob[31]:.3f}  open={o_prob[31]:.3f}')

In [ ]:
layers = np.arange(N_LAYERS)

# --- Figure 1: P(letter) by layer, all conditions ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(layers, s_prob, color='firebrick',  lw=2, label='stego')
ax.plot(layers, o_prob, color='steelblue',  lw=2, label='open')
ax.plot(layers, c_prob, color='darkorange', lw=2, label='control')
ax.axhline(1/26, color='grey', lw=1, ls='--', label='random baseline (1/26)')
ax.set_xlabel('Layer')
ax.set_ylabel('P(constrained letter)')
ax.set_title(f'exp13c: Logit lens — P(letter) by layer\n(n={n}, Llama-3.1-8B-Instruct)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/exp13c_logit_lens.png', dpi=150)
plt.show()

# --- Figure 2: Stego − Control difference per layer ---
diff_sc = s_prob - c_prob
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(layers, diff_sc,
       color=['firebrick' if d > 0 else 'steelblue' for d in diff_sc])
ax.axhline(0, color='black', lw=1)
ax.set_xlabel('Layer')
ax.set_ylabel('Δ P(letter)  [stego − control]')
ax.set_title(f'exp13c: Stego − Control per layer (KEY comparison)\n(n={n})')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/exp13c_stego_minus_control.png', dpi=150)
plt.show()

print('Figures saved.')

In [ ]:
summary = {
    'experiment': 'exp13c — logit lens: P(constrained letter) by layer',
    'n_triplets': n,
    'skipped': raw.get('skipped', 0),
    'logit_lens': {
        'stego_mean':   round(float(SA.mean()), 6),
        'open_mean':    round(float(OA.mean()), 6),
        'control_mean': round(float(CA.mean()), 6),
        'stego_vs_open':    {'diff': r_so[0], 't': r_so[1], 'p': r_so[2]},
        'stego_vs_control': {'diff': r_sc[0], 't': r_sc[1], 'p': r_sc[2]},
        'control_vs_open':  {'diff': r_co[0], 't': r_co[1], 'p': r_co[2]},
    },
    'key_layers': {
        'first_divergence': first_div,
        'L30': {'stego': round(float(s_prob[30]), 4), 'control': round(float(c_prob[30]), 4), 'open': round(float(o_prob[30]), 4)},
        'L31': {'stego': round(float(s_prob[31]), 4), 'control': round(float(c_prob[31]), 4), 'open': round(float(o_prob[31]), 4)},
    },
    'by_layer': {
        'stego':   [round(float(v), 6) for v in s_prob],
        'open':    [round(float(v), 6) for v in o_prob],
        'control': [round(float(v), 6) for v in c_prob],
        'stego_vs_control_pval': [round(float(p), 4) for p in layer_p_sc],
    },
}

out_path = f'{OUTPUT_DIR}/exp13c_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved:', out_path)
print(json.dumps({k: v for k, v in summary.items() if k != 'by_layer'}, indent=2))

In [ ]:
# Google Drive backup
if IN_COLAB and DRIVE_DIR:
    import pathlib
    for fp in pathlib.Path(OUTPUT_DIR).glob('exp13c*'):
        shutil.copy(fp, os.path.join(DRIVE_DIR, fp.name))
    nb = '/content/stego_CoT/notebooks/exp13c_logit_lens.ipynb'
    if os.path.exists(nb):
        shutil.copy(nb, os.path.join(DRIVE_DIR, 'exp13c_logit_lens.ipynb'))
    print(f'Backed up to {DRIVE_DIR}')